In [1]:
import os
import pandas as pd

In [7]:
shot_usage = pd.read_csv('./data/shot_usageNEW.csv', index_col=0)

In [8]:
distribution_df = pd.DataFrame(index=shot_usage.index, columns=['n_imgs_RIS1', 'n_imgs_RIS2'])

In [9]:
type(shot_usage.at[13182, 'used_for_ris1'])

numpy.bool_

In [10]:
for shot_number in shot_usage.index:
    for _, _, imgs in os.walk(f'./imgs/{shot_number}'):
        n_imgs_RIS1 = 0
        n_imgs_RIS2 = 0
        for img in imgs:
            if 'RIS1' in img and shot_usage.at[shot_number, 'used_for_ris1']:
                n_imgs_RIS1 += 1
            elif 'RIS2' in img and shot_usage.at[shot_number, 'used_for_ris2']:
                n_imgs_RIS2 += 1
        distribution_df.at[shot_number, 'n_imgs_RIS1'] = n_imgs_RIS1
        distribution_df.at[shot_number, 'n_imgs_RIS2'] = n_imgs_RIS2

In [11]:
distribution_df[(distribution_df['n_imgs_RIS1'] > 0) & (distribution_df['n_imgs_RIS2'] > 0)].shape

(56, 2)

In [12]:
non_zero_dist_df = distribution_df[~(distribution_df.any(axis=1) == 0)]

In [13]:
a = non_zero_dist_df.sort_values(by='n_imgs_RIS2', ascending=False)

In [14]:
non_zero_dist_df.sum()

n_imgs_RIS1    153440
n_imgs_RIS2    112291
dtype: object

In [15]:
# Set the directory path
directory = '/compass/Shared/Users/bogdanov/ml_tokamak/imgs'

# Initialize a counter for the number of files
file_count = 0

# Walk through all subfolders and count the files
for root, dirs, files in os.walk(directory):
    file_count += len(files)

# Print the total number of files
print(f'Total number of files: {file_count}')

Total number of files: 378675


In [16]:
root.split('/')[-1]

'19393'

### Mode distribution

In [33]:
from pathlib import Path
import confinement_mode_classifier as cmc
import numpy as np

ris_option = 'RIS1'

shot_usage = pd.read_csv(f'/compass/Shared/Users/bogdanov/ml_tokamak/data/shot_usageNEW.csv')
shot_for_ris = shot_usage[shot_usage['used_for_ris2'] if ris_option == 'RIS2' else shot_usage['used_for_ris1']]
shot_numbers = shot_for_ris['shot']
shots_for_testing = shot_for_ris[shot_for_ris['used_as'] == 'test']['shot']
shots_for_validation = shot_for_ris[shot_for_ris['used_as'] == 'val']['shot']
shots_for_training = shot_for_ris[shot_for_ris['used_as'] == 'train']['shot']

path = Path(os.getcwd())

In [34]:
shot_df, test_df, val_df, train_df = cmc.load_and_split_dataframes(path,shot_numbers, shots_for_training, shots_for_testing, 
                                                                    shots_for_validation, use_ELMS=True, ris_option=ris_option,
                                                                    exponential_elm_decay=False)

In [35]:
train_df['mode'].value_counts()

mode
0    64362
1    10385
2     6204
Name: count, dtype: int64

In [21]:

dist_df = pd.DataFrame({'train_df': train_df['mode'].value_counts().values, 'val_df': val_df['mode'].value_counts().values, 'test_df': test_df['mode'].value_counts().values}, 
                       index=['L-mode', 'H-mode', 'ELM'])


In [23]:
dist_df

,train_df,val_df,test_df
L-mode,64362,26814,23679
H-mode,10385,4278,4156
ELM,6204,3092,2670


In [26]:
dist_df.sum().sum()

145640

### How many continious ELM has ResNet recognized?

In [6]:
import os
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt
import io
from PIL import Image


predidctions_df = pd.read_csv('/compass/Shared/Users/bogdanov/ml_tokamak/runs/24-05-27, 20-00-44 both, finer lr_scheduler, 3 classes, resnet34_all_layers/prediction_df.csv', index_col=0)

In [7]:
predidctions_df

,shot,prediction,label,time,prob_0,prob_1,prob_2
0,16534,0,0,960.2,1.000000,3.376573e-08,8.351586e-08
1,16534,0,0,960.4,0.999934,5.506525e-06,6.088502e-05
2,16534,0,0,960.6,0.999980,1.946953e-05,1.545336e-07
3,16534,0,0,960.8,0.999846,1.525366e-04,1.709923e-06
4,16534,0,0,961.0,0.999998,1.859756e-06,4.890233e-07
...,...,...,...,...,...,...,...
109205,19379,0,0,1329.0,1.000000,7.890306e-09,1.695548e-10
109206,19379,0,0,1329.2,0.999991,9.325103e-06,2.766426e-10
109207,19379,0,0,1329.4,0.999997,2.654907e-06,2.526000e-07
109208,19379,0,0,1329.6,1.000000,1.639063e-07,1.729383e-10


In [12]:
thr = 0.3

def group_continuous_elms(df, thr=0.5):
    """
    Group consecutive ELM periods by shot and time continuity.
    Returns a list of tuples (shot, start_idx, end_idx, is_detected)
    """
    elm_groups = []
    
    for shot in df['shot'].unique():
        shot_df = df[df['shot'] == shot].sort_values('time').reset_index(drop=True)
        
        # Find continuous ELM periods (label = 2)
        elm_mask = shot_df['label'] == 2
        if not elm_mask.any():
            continue
            
        # Find start and end of each continuous ELM period
        elm_changes = elm_mask.diff().fillna(False)
        elm_starts = shot_df.index[elm_changes & elm_mask].tolist()
        elm_ends = shot_df.index[elm_changes & ~elm_mask].tolist()
        
        # Handle case where ELM period extends to end of shot
        if elm_mask.iloc[-1] and (not elm_ends or elm_ends[-1] < elm_starts[-1]):
            elm_ends.append(len(shot_df))
        
        # Process each ELM group
        for start, end in zip(elm_starts, elm_ends):
            elm_group = shot_df.iloc[start:end]
            total_elms = len(elm_group)
            correct_predictions = (elm_group['prediction'] == 2).sum()
            
            # Check if at least 2/3 are correctly predicted
            is_detected = correct_predictions >= (thr * total_elms)
            elm_groups.append((shot, start, end, is_detected, total_elms, correct_predictions))
    
    return elm_groups

# Group continuous ELMs
elm_groups = group_continuous_elms(predidctions_df, thr=thr)

# Calculate metrics
total_elm_groups = len(elm_groups)
detected_elm_groups = sum(1 for group in elm_groups if group[3])  # group[3] is is_detected

print(f"Total continuous ELM groups found: {total_elm_groups}")
print(f"ELM groups detected (≥{thr:.1f} correct): {detected_elm_groups}")

# Calculate precision, recall, and F1
# For this analysis, we consider:
# - True Positives: ELM groups that were detected (≥2/3 correct predictions)
# - False Negatives: ELM groups that were not detected (<2/3 correct predictions)
# - False Positives: Need to find predicted ELM groups that don't correspond to actual ELMs

def group_predicted_elms(df, thr=0.5):
    """Group consecutive predicted ELM periods."""
    predicted_elm_groups = []
    
    for shot in df['shot'].unique():
        shot_df = df[df['shot'] == shot].sort_values('time').reset_index(drop=True)
        
        # Find continuous predicted ELM periods (prediction = 2)
        pred_elm_mask = shot_df['prediction'] == 2
        if not pred_elm_mask.any():
            continue
            
        # Find start and end of each continuous predicted ELM period
        pred_changes = pred_elm_mask.diff().fillna(False)
        pred_starts = shot_df.index[pred_changes & pred_elm_mask].tolist()
        pred_ends = shot_df.index[pred_changes & ~pred_elm_mask].tolist()
        
        # Handle case where predicted ELM extends to end
        if pred_elm_mask.iloc[-1] and (not pred_ends or pred_ends[-1] < pred_starts[-1]):
            pred_ends.append(len(shot_df))
        
        # Check each predicted group against actual labels
        for start, end in zip(pred_starts, pred_ends):
            pred_group = shot_df.iloc[start:end]
            actual_elms = (pred_group['label'] == 2).sum()
            total_predictions = len(pred_group)
            
            # Consider it a true positive if ≥2/3 of the predicted group overlaps with actual ELMs
            is_true_positive = actual_elms >= (thr * total_predictions)
            predicted_elm_groups.append((shot, start, end, is_true_positive, total_predictions, actual_elms))
    
    return predicted_elm_groups

predicted_elm_groups = group_predicted_elms(predidctions_df, thr=thr)

# Calculate final metrics
true_positives = sum(1 for group in predicted_elm_groups if group[3])
false_positives = len(predicted_elm_groups) - true_positives
false_negatives = total_elm_groups - detected_elm_groups

precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"\nContinuous ELM Detection Metrics:")
print(f"True Positives: {true_positives}")
print(f"False Positives: {false_positives}")
print(f"False Negatives: {false_negatives}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1_score:.3f}")

# Additional analysis: distribution of ELM group sizes and detection rates
elm_group_info = pd.DataFrame(elm_groups, columns=['shot', 'start', 'end', 'detected', 'total_elms', 'correct_preds'])
elm_group_info['group_size'] = elm_group_info['end'] - elm_group_info['start']
elm_group_info['detection_rate'] = elm_group_info['correct_preds'] / elm_group_info['total_elms']

print(f"\nELM Group Statistics:")
print(f"Average ELM group size: {elm_group_info['group_size'].mean():.1f}")
print(f"Median ELM group size: {elm_group_info['group_size'].median():.1f}")
print(f"Average detection rate within groups: {elm_group_info['detection_rate'].mean():.3f}")

# Show detection rate by group size
size_analysis = elm_group_info.groupby('group_size').agg({
    'detected': ['count', 'sum', 'mean']
}).round(3)
print(f"\nDetection rate by ELM group size:")
print(size_analysis.head(10))

/tmp/ipykernel_61775/4282976074.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  elm_changes = elm_mask.diff().fillna(False)
/tmp/ipykernel_61775/4282976074.py:68: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pred_changes = pred_elm_mask.diff().fillna(False)


Total continuous ELM groups found: 847
ELM groups detected (≥0.3 correct): 783

Continuous ELM Detection Metrics:
True Positives: 1188
False Positives: 392
False Negatives: 64
Precision: 0.752
Recall: 0.949
F1 Score: 0.839

ELM Group Statistics:
Average ELM group size: 8.4
Median ELM group size: 7.0
Average detection rate within groups: 0.809

Detection rate by ELM group size:
           detected            
              count  sum   mean
group_size                     
1                10    0  0.000
2                15    3  0.200
3                16   15  0.938
4                74   63  0.851
5               134  133  0.993
6               148  143  0.966
7               166  166  1.000
8                57   53  0.930
9                 6    6  1.000
10               37   33  0.892

Continuous ELM Detection Metrics:
True Positives: 1188
False Positives: 392
False Negatives: 64
Precision: 0.752
Recall: 0.949
F1 Score: 0.839

ELM Group Statistics:
Average ELM group size: 8.4
Median EL